In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 04 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 04.C1 — Compose permutations

Represent a permutation by p[i], the image of i. Return p after q: apply q first, then p.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The right-hand operation acts first in this convention.

</details>

In [3]:
def compose(p,q):
    return tuple(p[q[i]] for i in range(len(q)))

In [4]:
r=(1,2,3,0);s=(0,3,2,1)
assert compose(r,s)!=(compose(s,r))
assert compose(r,(0,1,2,3))==r
print("Rotation after reflection:",compose(r,s),"reverse:",compose(s,r))

Rotation after reflection: (1, 0, 3, 2) reverse: (3, 2, 1, 0)


**Why this works and what it does not establish:** Function composition is associative but need not commute. Recording the convention prevents an apparent disagreement caused only by reversing the order of application.

## 04.C2 — Invert a permutation

For a valid permutation of 0..n-1 return the inverse tuple.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

If p sends i to j, the inverse sends j back to i.

</details>

In [5]:
def invert_permutation(p):
    inverse=[0]*len(p)
    for i,value in enumerate(p): inverse[value]=i
    return tuple(inverse)

In [6]:
p=(1,2,3,0);q=invert_permutation(p)
assert tuple(p[q[i]] for i in range(4))==(0,1,2,3)
assert q==(3,0,1,2)
print("Inverse:",q)

Inverse: (3, 0, 1, 2)


**Why this works and what it does not establish:** An inverse undoes the operation, not the numerical values stored in the representation. A rotation inverse is a rotation in the opposite direction.

## 04.C3 — Rotate a real image through a group action

Return an image rotated k quarter-turns counterclockwise. Verify that four quarter-turns restore exactly the same array.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

np.rot90 rotates the array; k counts quarter-turns.

</details>

In [7]:
def quarter_turn(image,k):
    return np.rot90(np.asarray(image),k=k)

In [8]:
image,sample_id=digit_example(6)
assert np.array_equal(quarter_turn(image,4),image)
assert np.array_equal(quarter_turn(quarter_turn(image,1),1),quarter_turn(image,2))
print("C4 action checked on training image",sample_id)

C4 action checked on training image 16


**Why this works and what it does not establish:** The array transformation is exact, but the class meaning need not be invariant: rotating handwriting can change its interpretation. Mathematical symmetry and a task-appropriate data augmentation are different claims.

## 04.C4 — Write an annulus deformation

For a nonzero point x in an annulus around the origin, return H(x,t)=((1-t)+t/||x||)x for 0<=t<=1.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The norm rescales direction to unit length. The domain excludes the origin.

</details>

In [9]:
def radial_homotopy(x,t):
    x=np.asarray(x,dtype=float)
    return ((1-t)+t/np.linalg.norm(x))*x

In [10]:
assert np.allclose(radial_homotopy([2,0],0),[2,0])
assert np.allclose(radial_homotopy([2,0],1),[1,0])
for t in np.linspace(0,1,7): assert np.allclose(radial_homotopy([0,1],t),[0,1])
print("Start, end, and fixed-circle conditions checked")

Start, end, and fixed-circle conditions checked


**Why this works and what it does not establish:** These checks illustrate endpoints and the fixed subspace. The continuous formula and its domain justify the deformation retraction; a finite set of successful tests cannot by itself prove continuity.

## 04.C5 — Reduce a word with inverse pairs

Use lowercase a,b and uppercase A,B for formal generators and inverses. Cancel adjacent inverse pairs until no more remain.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The top of a stack is the only possible partner for the next cancellation.

</details>

In [11]:
def reduce_word(word):
    stack=[]
    for letter in word:
        if stack and stack[-1].swapcase()==letter: stack.pop()
        else: stack.append(letter)
    return ''.join(stack)

In [12]:
assert reduce_word('abBA')==''
assert reduce_word('abAB')=='abAB'
assert reduce_word('aaA')=='a'
print("A commutator does not cancel in the free-word model:",reduce_word('abAB'))

A commutator does not cancel in the free-word model: abAB


**Why this works and what it does not establish:** This is a formal free-group word model. It shows why loop order can matter in spaces such as a figure-eight. It does not establish the fundamental group of every pictured shape.